# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [30]:
# imports
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [31]:
# constants

MODEL_GPT = "gpt-4o-mini"
MODEL_GPT_OPEN_SOURCE = "gpt-oss"
MODEL_LLAMA = "llama3.2"
MODEL_LLAMA_BASE_URL = os.getenv("MODEL_BASE_URL", "http://localhost:11434/v1")


SYSTEM_PROMPT = """
You are an expert technical educator.

Answer the user's technical question clearly, accurately, and concisely.

- Explain technical terms when necessary.
- Break complex concepts into simple parts.
- For code, explain what it does and why it works.
- Use examples when helpful.
- Do not invent information. If uncertain, say so.
- Do not give a long answer unless the user asks for one.

Respond in Markdown.
"""

In [32]:
# set up environment
load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if (
    OPENAI_API_KEY
    and len(OPENAI_API_KEY) > 10
    and OPENAI_API_KEY.startswith("sk-proj-")
):
    print("OPENAI_API_KEY looks good so far.")
else:
    print(
        "There might be a problem with your API key? Please visit the troubleshooting notebook!"
    )

OPENAI_API_KEY looks good so far.


In [33]:
# here is the question; type over this to ask something new

question = """
Please explain what is streaming in openai objet
"""

In [37]:
def stream_technical_question_answer(client, model_name, question):
    stream = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ],
        stream=True,
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        update_display(Markdown(response), display_id=display_handle.display_id)

    # return response

In [35]:
# Get gpt-4o-mini to answer, with streaming

# uncomment below 2 lines, for calling openai API, if you have credits in openai account
# model_gpt_client = OpenAI() 
# stream_technical_question_answer(client=model_llama_client, model_name=MODEL_LLAMA, question=question)

model_gptoss_client = OpenAI(base_url=MODEL_LLAMA_BASE_URL, api_key="ollama") # else try this, this will work without any need of credit card
stream_technical_question_answer(client=model_gptoss_client, model_name=MODEL_GPT_OPEN_SOURCE, question=question)

## Streaming with the OpenAI API

**Streaming** is an option you can enable when calling a model (e.g., `ChatCompletion`, `TextCompletion`).  
Instead of waiting for the **whole response to finish**, the server sends back pieces (chunks) of the answer as soon as they’re generated.

### How it works

| Step | What happens |
|------|--------------|
| 1. Request | You send a request with the parameter `stream=True` (or `stream: true` in Node). |
| 2. Generation | The model starts generating tokens one by one. |
| 3. Streaming | Each token (or small group of tokens) is wrapped in a JSON “chunk” and sent via **Server‑Sent Events (SSE)** over the same HTTP connection. |
| 4. Reception | Your client parses each chunk as it arrives, appends it to the final result, and optionally displays it live. |
| 5. End | A final chunk with `is_last: true` signals that generation is complete. The HTTP stream closes.

### Key benefits

- **Lower latency** – you can start reading output before the API finishes.
- **Real‑time UI updates** – useful for chatbots, live text editors, or any interactive app.
- **Reduced memory usage** – your backend doesn’t have to buffer an entire long response first.

### What a streaming chunk looks like (Python example)

```python
import openai

response = openai.ChatCompletion.create(
    model="gpt-4o-mini",
    messages=[{"role":"user","content":"Tell me about quantum computing."}],
    stream=True          # ← enable streaming
)

for chunk in response:
    # Each `chunk` is a dictionary. For completions, the relevant part lives in:
    token = chunk["choices"][0]["delta"].get("content")
    if token:                                 # None for context-only events
        print(token, end="", flush=True)      # live display
```

A simplified SSE payload looks like:

```json
{
  "id": "chatcmpl-xxxx",
  "object": "chat.completion.chunk",
  "created": 1712226665,
  "model": "gpt-4o-mini",
  "choices": [
    {
      "index": 0,
      "delta": { "content": "Quantum" },
      "finish_reason": null
    }
  ]
}
```

The `delta.content` field contains the new token text. When the last chunk arrives, `finish_reason` will be `"stop"` or `"length"`.

### Typical use‑case

1. **Chat applications** – show response words as they appear.  
2. **Command‑line tools** – stream progress for long answers.  
3. **Streaming data pipelines** – process outputs incrementally without buffering.

---

### Bottom line

- **Streaming** = getting model output *in real time* in small chunks, rather than a single big reply.  
- It’s activated by setting `stream=True` and handled with an incremental loop on the client side.  
- This mode is ideal for interactive experiences where waiting minutes for a full reply would feel sluggish.

'## Streaming with the OpenAI API\n\n**Streaming** is an option you can enable when calling a model (e.g., `ChatCompletion`, `TextCompletion`).  \nInstead of waiting for the **whole response to finish**, the server sends back pieces (chunks) of the answer as soon as they’re generated.\n\n### How it works\n\n| Step | What happens |\n|------|--------------|\n| 1. Request | You send a request with the parameter `stream=True` (or `stream: true` in Node). |\n| 2. Generation | The model starts generating tokens one by one. |\n| 3. Streaming | Each token (or small group of tokens) is wrapped in a JSON “chunk” and sent via **Server‑Sent Events (SSE)** over the same HTTP connection. |\n| 4. Reception | Your client parses each chunk as it arrives, appends it to the final result, and optionally displays it live. |\n| 5. End | A final chunk with `is_last: true` signals that generation is complete. The HTTP stream closes.\n\n### Key benefits\n\n- **Lower latency** – you can start reading output bef

In [38]:
# Get Llama 3.2 to answer

model_llama_client = OpenAI(base_url=MODEL_LLAMA_BASE_URL, api_key="ollama")

stream_technical_question_answer(client=model_llama_client, model_name=MODEL_LLAMA, question=question)

**Streaming in OpenAI Object Detection**
==========================================

In computer vision, **streaming** refers to the process of processing video or image sequences without storing them in memory. Instead, the data is transmitted in real-time, allowing for efficient and scalable object detection applications.

**Why Streaming?**

OpenAI's Object Detector relies on this streaming approach to achieve various advantages:

*   **Scalability**: By not storing large amounts of images, you can process many frames from a video file without exhausting memory resources.
*   **Faster Processing**: Stream processing enables the application to react more quickly, which is beneficial for real-world applications like self-driving cars or surveillance systems.

**How Streaming Works**
-----------------------

Stream processing typically involves:

1.  **Data Ingestion**: Receiving the raw input data through an interface (e.g., video file) in real-time.
2.  **Preprocessing**: Performing initial computations necessary to extract meaningful features from the input data, such as resizing and normalizing images.
3.  **Object Detection**: Using a neural network-based detector like OpenAI's Object Detector, which analyzes the preprocessed data and identifies objects.
4.  **Postprocessing**: Processing and displaying detected object bounding boxes on the original image.

**Example: OpenAI's Object Detector in Code**
--------------------------------------------

Here is some simplified code illustrating how OpenAI's Object Detector can be used for stream processing:
```python
import cv2
from openai import detector_v3

# Initialize the detector model
det_model = detector_v3.DetectorV3()

def process_video(video_file):
    # Load video file and convert it into a stream object
    cap = cv2.VideoCapture(video_file)

    while True:
        ret, frame = cap.read()
        
        if not ret:
            break
        
        # Apply image preprocessing (resize to 640, normalize)
        resized_frame = cv2.resize(frame, (640, 480))
        resized_frame /= 255
        
        # Detect objects
        results = det_model.detect(resized_frame)[0]
        
        # Process and display detected bounding boxes ('ROI')
        for ROI in results['data']:
            x1, y1, x2, y2 = ROI.get(x1=True), ROI.get(y1=True), ROI.get(x2=True), ROI.get(y2=True)
            cv2.rectangle(frame, (x1,y1),(x2,y2),(0,255,0), 2)

            # Draw object in the original image
        cv2.imshow('frame', frame)
        
        if cv2.waitKey(1)&0xFF == ord('q'):
            break
    
    cap.release()
    
# Run video on a webcam 
from PiCamera import Camera
piCamera = Camera(640,480)
process_video("your_viedo.mp4")
```